<a href="https://colab.research.google.com/github/organichotdog/micrograd/blob/main/micrograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
class Value:

  def __init__(self, value, children = (), operation = '', grad = 0, exponent = 1):
    self.data = value
    self.children = children
    self.operation = operation
    self.grad = grad
    self.exponent = exponent


  def __repr__(self):
    return f"{self.data}, {self.children}, {self.operation}, {self.grad}"


  def __add__(self, val):
    val = Value(val) if not isinstance(val, Value) else val
    return Value(
        self.data + val.data,
        children=(self, val),
        operation='+'
    )


  def __radd__(self, val):
    return self + val


  def __mul__(self, val):
    val = Value(val) if not isinstance(val, Value) else val
    return Value(
        self.data * val.data,
        children=(self, val),
        operation='*'
    )


  def __rmul__(self, val):
    return self * val


  def __pow__(self, exp):
    return Value(
        self.data ** exp,
        children=(self,),
        operation='**',
        exponent = exp
    )


  def ordering(self, graph):
    if self not in graph:
      graph.append(self)
      for child in self.children:
        child.ordering(graph)
      return graph


  def backward(self):
    if self.operation == '+':
      self.children[0].grad += self.grad
      self.children[1].grad += self.grad
    elif self.operation == '*':
      self.children[0].grad += self.children[1].data * self.grad
      self.children[1].grad += self.children[0].data * self.grad
    elif self.operation == "**":
      self.children[0].grad += self.exponent * (self.children[0].data ** (self.exponent - 1)) * self.grad


In [38]:
a = Value(3)
b = a * 3

print(b)

b.grad = 1

graph = []

sorted_graph = b.ordering(graph)

for node in sorted_graph:
  node.backward()

print(a.grad)
print(b.grad)

9, (3, (), , 0, 3, (), , 0), *, 0
3
1
